# Challenge 01 — conditional EGFR binder

Objectives, in the stated priority order:

1. pH selectivity — binds at pH 6.5, releases at pH 7.4
2. Mouse cross-reactivity
3. Affinity for the human domain III functional epitope

Constraint: de novo and zero-shot. No existing binder used as a seed.

The forward models are pH-blind, so objective 1 gets no signal from folding.
The pH signal is supplied downstream by a thermodynamic linkage score over
predicted pKa shifts, with a hard geometric filter underneath it.



In [1]:
from pathlib import Path

import pandas as pd

from binderkit import epitope, liability, phscore, select, submission

ROOT = Path.cwd()
INPUTS = ROOT / "inputs"
OUTPUTS = ROOT / "outputs"
OUTPUTS.mkdir(exist_ok=True)

TARGET_STRUCTURE = INPUTS / "target_chainA.pdb"
HUMAN_FASTA = INPUTS / "target_human.fasta"
MOUSE_FASTA = INPUTS / "target_mouse.fasta"
REFERENCE_FASTA = INPUTS / "reference_binders.fasta"  # novelty set, optional

TARGET_CHAIN = "A"
BINDER_CHAIN = "B"
DOMAIN_III = (310, 480)  # residue window on the target chain, author numbering
PH_LOW, PH_HIGH = 6.5, 7.4

## 1. Epitope



In [ ]:
from Bio import Align

def _pairwise_aligner_callable(self, *args, **kwargs):
    return self.align(*args, **kwargs)

Align.PairwiseAligner.__call__ = _pairwise_aligner_callable

records = epitope.build_records(
    structure_path=str(TARGET_STRUCTURE),
    chain_id=TARGET_CHAIN,
    human_fasta=str(HUMAN_FASTA),
    mouse_fasta=str(MOUSE_FASTA),
    domain_range=DOMAIN_III,
)

anchors = epitope.anchor_residues(records, sasa_cut=0.25)
print(f"residues in window: {len(records)}")
print(f"conserved + acidic + exposed anchors: {len(anchors)}")
[f"{r.resname}{r.resnum} rel_sasa={r.rel_sasa:.2f}" for r in anchors]

residues in window: 171
conserved + acidic + exposed anchors: 7


['GLU320 rel_sasa=0.48',
 'ASP323 rel_sasa=0.71',
 'ASP364 rel_sasa=0.43',
 'GLU367 rel_sasa=0.28',
 'GLU400 rel_sasa=0.43',
 'GLU431 rel_sasa=0.35',
 'GLU472 rel_sasa=0.46']

In [4]:
patches = epitope.rank_patches(records, radius=12.0, sasa_cut=0.25, min_anchors=2)
patch_df = pd.DataFrame(patches).drop(columns=["hotspot_resnums"], errors="ignore")
patch_df.head(10)

,centre,centre_resnum,n_anchors,anchors,n_residues,conserved_fraction,mean_rel_sasa
0,GLU320,320,2,"[GLU320, ASP323]",11,1.000,0.479
1,ASP323,323,2,"[GLU320, ASP323]",8,1.000,0.492
2,GLU400,400,2,"[GLU400, GLU431]",6,1.000,0.366
3,GLU431,431,2,"[GLU400, GLU431]",9,1.000,0.463
4,GLU367,367,2,"[ASP364, GLU367]",11,0.818,0.430
5,ASP364,364,2,"[ASP364, GLU367]",10,0.700,0.453


In [5]:
FOOTPRINT = {349,350,353,382,384,408,409,411,412,415,417,418,438,440,441,443,465,466,467,468,469,471,472,473}
for p in patches:
    overlap = sorted(set(p["hotspot_resnums"]) & FOOTPRINT)
    print(p["centre"], p["n_residues"], f"overlap={len(overlap)}", overlap)

GLU320 11 overlap=0 []
ASP323 8 overlap=0 []
GLU400 6 overlap=0 []
GLU431 9 overlap=0 []
GLU367 11 overlap=0 []
ASP364 10 overlap=1 [353]


In [6]:
import numpy as np
by_num = {r.resnum: r for r in records}
foot = np.array([by_num[n].ca_coord for n in FOOTPRINT if n in by_num])
for p in patches:
    c = by_num[p["centre_resnum"]].ca_coord
    print(p["centre"], round(float(np.linalg.norm(foot - c, axis=1).min()), 1))

GLU320 18.5
ASP323 14.2
GLU400 15.7
GLU431 14.4
GLU367 14.6
ASP364 10.2


In [7]:
sel = {p["centre"]: p for p in patches}
for key in ("GLU320", "GLU367"):
    p = sel[key]
    print(key, epitope.hotspot_string(p, TARGET_CHAIN))

GLU320 A314,A317,A318,A319,A320,A322,A323,A325,A330,A333,A334
GLU367 A330,A333,A334,A336,A362,A364,A365,A366,A367,A369,A372


In [8]:
import numpy as np
print(round(float(np.linalg.norm(by_num[320].ca_coord - by_num[367].ca_coord)), 1))

18.4


In [9]:
union = sorted(set(sel["GLU320"]["hotspot_resnums"]) | set(sel["GLU367"]["hotspot_resnums"]))
hotspots = ",".join(f"{TARGET_CHAIN}{n}" for n in union)
print(len(union), hotspots)
pd.DataFrame(patches).to_csv(OUTPUTS / "epitope_patches.csv", index=False)

19 A314,A317,A318,A319,A320,A322,A323,A325,A330,A333,A334,A336,A362,A364,A365,A366,A367,A369,A372


In [ ]:
chosen = patches[0]
hotspots = epitope.hotspot_string(chosen, TARGET_CHAIN)
print(chosen["centre"], chosen["n_anchors"], chosen["conserved_fraction"])
print(hotspots)

pd.DataFrame(patches).to_csv(OUTPUTS / "epitope_patches.csv", index=False)

## 2. Generation and sequence design

Run externally against the hotspot string above. Histidines are fixed at the
positions facing the patch carboxylates during sequence design rather than left
to the sampler, since the designer has no pH-aware objective to find them with.

Record the command, the version and the seed here, then drop the predicted
complexes into `outputs/complexes/` with one PDB per design, binder as chain
`B` and target as chain `A`.

In [ ]:
COMPLEX_DIR = OUTPUTS / "complexes"
complexes = sorted(COMPLEX_DIR.glob("*.pdb"))
len(complexes)

## 3. pH scoring

Two pKa runs per design, free binder and complex, restricted to interface
ionisable sites. `geometry_pass` is the hard filter: no histidine nitrogen
within salt-bridge distance of a target carboxylate means no mechanism, and
the design is dropped whatever its score. `sign_pass` keeps only designs whose
binding is more favourable at the lower pH.

In [ ]:
ph_rows = []
for path in complexes:
    result = phscore.score_design(
        complex_path=path,
        binder_chain=BINDER_CHAIN,
        target_chain=TARGET_CHAIN,
        ph_low=PH_LOW,
        ph_high=PH_HIGH,
        min_his_contacts=2,
        workdir=OUTPUTS / "propka" / path.stem,
    )
    ph_rows.append({k: v for k, v in result.items() if k not in ("per_site", "contacts")})

ph_df = pd.DataFrame(ph_rows)
ph_df.to_csv(OUTPUTS / "ph_scores.csv", index=False)
ph_df.sort_values("ph_selectivity_kcal").head(20)

## 4. Fold metrics

Load the ipTM, interface PAE and pLDDT written by the structure predictor for
the human complex, and the same metrics for the re-folded mouse complex. These
come from the prediction run, not from this package.

In [ ]:
fold_df = pd.read_csv(OUTPUTS / "fold_metrics.csv")
fold_df.head()

## 5. Liabilities and novelty

Hard filters, not ranking signals. Free cysteines, aggregation-prone surface
patches, expression-liability motifs, length bounds, and identity against the
reference set for the de novo requirement.

In [ ]:
sequences = pd.read_csv(OUTPUTS / "designs.csv")  # columns: design, sequence

liability_rows = []
for _, row in sequences.iterrows():
    structure = COMPLEX_DIR / f"{row['design']}.pdb"
    report = liability.assess(
        name=row["design"],
        sequence=row["sequence"],
        structure_path=structure if structure.exists() else None,
        chain_id=BINDER_CHAIN,
        reference_fasta=REFERENCE_FASTA if REFERENCE_FASTA.exists() else None,
    )
    liability_rows.append(report.as_row())

liability_df = pd.DataFrame(liability_rows).rename(columns={"name": "design"})
liability_df.to_csv(OUTPUTS / "liabilities.csv", index=False)
liability_df["passes"].value_counts()

## 6. Merge, Pareto front, diversity pick

Objectives are maximised after sign correction: pH selectivity is negated
because a more negative linkage difference is the wanted direction. Hard
filters are applied before ranking, never traded against it.

In [ ]:
merged = (
    sequences.merge(ph_df.rename(columns={"design": "design"}), on="design", how="inner")
    .merge(fold_df, on="design", how="inner")
    .merge(liability_df, on="design", how="inner")
)
merged["epitope_patch"] = chosen["centre"]
merged.to_csv(OUTPUTS / "merged_metrics.csv", index=False)
merged.shape

In [ ]:
objectives = {
    "ph_selectivity_kcal": "min",   # more negative is better
    "iptm_mouse": "max",
    "iptm_human": "max",
}

picked = select.select_submission(
    merged,
    objectives=objectives,
    n=20,
    hard_filters=["ph_pass", "passes"],
    primary_objective="ph_selectivity_kcal",
    min_distance=0.4,
)
picked[["design", "ph_selectivity_kcal", "iptm_human", "iptm_mouse", "pareto_rank"]]

## 7. Submission

In [ ]:
table = submission.build(
    picked,
    name_column="design",
    sequence_column="sequence",
    molecule_class="protein",
    name_prefix="nd-egfr-ph",
)

result = submission.write(table, ROOT / "submission.csv")
print(result.report())
table